In [2]:
from copy import copy
import numpy as np
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots
from scipy.optimize import bisect, brentq
from scipy.stats import norm

In [3]:
class CallOption:

    def __init__(self, K, T, price=None):
        self.K = K
        self.T = T
        self.price = price

In [4]:
class GBMdynamics:

    def __init__(self, X, r, rGrow, sigma=None):
        self.X = X
        self.r = r
        self.rGrow = rGrow
        self.sigma = sigma

    def update_sigma(self, sigma):
        self.sigma = sigma
        return self

In [5]:
class AnalyticEngine:

    def __init__(self):
        pass

    def BSpriceCall(self, dynamics, contract):
        # Ignores contract.price if given, because this function calculates price based on the dynamics.
        # Returns time-0 price.

        F = dynamics.X*np.exp(dynamics.rGrow*contract.T)
        std = dynamics.sigma*np.sqrt(contract.T)
        d1 = np.log(F/contract.K)/std+std/2
        d2 = d1-std
        return np.exp(-dynamics.r*contract.T)*(F*norm.cdf(d1)-contract.K*norm.cdf(d2))

    def BSvega(self, dynamics, contract):
        # Returns time-$0$ vega

        F = dynamics.X*np.exp(dynamics.rGrow*contract.T)
        std = dynamics.sigma*np.sqrt(contract.T)
        d1 = np.log(F/contract.K)/std+std/2
        return np.exp(-dynamics.r*contract.T)*F*norm.pdf(d1)*np.sqrt(contract.T)

    def IV(self, dynamics, contract):
        # ignores dynamics.sigma, because this function solves for sigma.
        # Returns time-$0$ implied volatility

        if contract.price is None:
            raise ValueError('Contract price must be given')

        df = np.exp(-dynamics.r*contract.T)  #discount factor
        F = dynamics.X*np.exp(dynamics.rGrow*contract.T)
        lowerbound = np.max([0, df*(F - contract.K)])
        C = contract.price
        if C < lowerbound:
            return np.nan
        if C == lowerbound:
            return 0
        if C >= F*df:
            return np.nan

        dynamics_try = copy(dynamics)
        # We "try" values of sigma until we find sigma that generates price C

        # First find lower and upper bounds
        sigma_try = 0.2
        while self.BSpriceCall(dynamics_try.update_sigma(sigma_try), contract) > C:
            sigma_try /= 2
        while self.BSpriceCall(dynamics_try.update_sigma(sigma_try), contract) < C:
            sigma_try *= 2
        hi = sigma_try
        lo = hi/2
        # We have calculated "lo" and "hi" which bound the implied volatility from below and above.
        # In other words, the implied volatility is somewhere in the interval [lo,hi].
        # Then, to calculate the implied volatility within that interval,
        # for purposes of this homework, you may either (A) write your own bisection algorithm,
        # or (B) use scipy.optimize.bisect or (C) use scipy.optimize.brentq
        # You will need to provide lo and hi to those solvers.
        # There are other solvers that do not require you to bound the solution
        # from below and above (for instance, scipy.optimize.fsolve is a useful solver).
        # However, if you are able to bound the solution (of a single-variable problem),
        # then bisection or Brent will be more reliable.

        # Use bisect or brentq imported from scipy.optimize, or write your own bisection algorithm
        impliedVolatility = # FILL THIS IN

        return impliedVolatility

SyntaxError: invalid syntax (2681689769.py, line 65)

## Problem 1

In [ ]:
options = pd.read_csv('hw1-rut.csv')   #You may need to provide a path to the location of this file
options.set_index('Strike', drop=False, inplace=True)

T = 25/365
r = 0.0432

In [ ]:
# Calculate the forward price.
F =   # Fill this in

In [ ]:
options['ImpliedCallMid'] = # Fill this in with an "implied" call price obtained from the (midpoint) put price at each strike

In [ ]:
options['CallOrImpliedCall'] = # At strikes where the call is OTM, assign this to be the (midpoint) call price.
                               # At strikes where the put is OTM, assign this to be the implied (midpoint) call price, obtained from the (midpoint) put price

In [ ]:
hw1analytic = AnalyticEngine()
dynamics = GBMdynamics(X=F, r=r, rGrow=0, sigma=None)

In [ ]:
options['IV'] = options.apply(lambda row: hw1analytic.IV(dynamics,  CallOption(K=row.Strike, T=T, price=row.CallOrImpliedCall)), axis=1)

In [ ]:

fig = go.Figure()

fig.add_trace(go.Scatter(x=options.Strike, y=options.IV, mode='markers', name='Implied Volatility'))

fig.update_layout(title='Implied Volatility vs. Strike',
                  xaxis_title='Strike Price',
                  yaxis_title='Implied Volatility')
fig.show()


##Problem 2

In [ ]:

k = np.log(options.Strike / F)

coefficients = # Fill this in by using a function such as numpy.polyfit to solve for array of coefficients (alpha0, ... , alpha4)
polynomial = np.poly1d(coefficients)


In [ ]:
k_grid = np.linspace(k.min(), k.max(), 100)
y_fit = polynomial(k_grid)

fig = go.Figure()

fig.add_trace(go.Scatter(x=k, y=options.IV, mode='markers', name='Observed midpoint implied volatility'))
fig.add_trace(go.Scatter(x=k_grid, y=y_fit, mode='lines', name='Fitted polynomial parameterization'))

fig.update_layout(title='Implied Volatility vs log-moneyness',
                  xaxis_title='log-moneyness',
                  yaxis_title='Implied Volatility')

fig.show()

In [ ]:
# Fill this in:
# overpricedStrike =

In [ ]:
# The vega calculation for each strike should use the fitted polynomial implied vol for that strike
options['vega'] = options.apply(lambda row: hw1analytic.BSvega(GBMdynamics( ## Fill this in with appropriate parameters
                                                                           ), CallOption(K=row.Strike, T=T)), axis=1)

In [ ]:
options['skewsensitivity'] = #Fill this in

In [ ]:
import plotly.express as px

fig = px.line(options, x="Strike", y="skewsensitivity", title='Sensitivity to Skew Slope, vs. Strike')
fig.show()

In [ ]:
whichContractGainsMostFromSlopeIncrease = # Fill this in

In [ ]:
whichContractLosesMostFromSlopeIncrease = # Fill this in